In [97]:
import re
import requests
import random
import time
import requests
import math
import glob
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Dict
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
from xgboost import XGBRegressor

In [ ]:
def download(url):
    """Download a page with basic retry."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}
    for _ in range(3):
        try:
            txt = requests.get(url, headers=headers, timeout=10)
            if txt.status_code == 200:
                return txt.text
            else:
                print(f"[WARN] status_code={r.status_code} for {url}")
        except Exception as e:
            print(f"[WARN] download error for {url}: {e}")
    return ""


def extract_line_urls_from_ensen(html):
    """
    Extract all Benchmark Portal line URLs from the /ensen/ page.
    Return a list of full URLs.
    Print duplicate count (as required).
    """

    # regex for Benchmark Portal line entries
    pattern = r'href="(/chintai/soba/tokyo/en_[a-z0-9]+/)"'

    found = re.findall(pattern, html)

    full_urls = []
    seen = set()
    dup = 0

    for path in found:
        url = "https://www.example-realestate-benchmark.jp" + path
        if url in seen:
            dup += 1
        else:
            seen.add(url)
            full_urls.append(url)

    print(f"Line URLs extracted: {len(full_urls)}  dup={dup}")

    return full_urls
# regex patterns (based strictly on your uploaded Benchmark Portal HTML)
RE_ROW = re.compile(
    r'<tr class="js-graph-data"[^>]*data-value="([0-9.]+)"[^>]*>(.*?)</tr>',
    re.DOTALL
)

RE_STATION = re.compile(
    r'<td>(?:<a[^>]*>)?([^<]+)(?:</a>)?</td>'
)

def process_single_line(url, dic):
    html = download(url)

    # extract the canonical line ID like "en_yamanotesen"
    m = re.search(r'/chintai/soba/tokyo/([^/]+)/', url)
    line_id = m.group(1)

    new_cnt = 0
    dup_cnt = 0
    skip_cnt = 0

    rows = RE_ROW.findall(html)

    for val_str, row_html in rows:
        try:
            yen = int(float(val_str) * 10000)
        except:
            skip_cnt += 1
            continue

        m2 = RE_STATION.search(row_html)
        if not m2:
            skip_cnt += 1
            continue

        station = m2.group(1).strip()

        # write into dictionary
        if station in dic:
            dup_cnt += 1
        else:
            new_cnt += 1

        dic[station] = yen

    print(f"{line_id} new={new_cnt} dup={dup_cnt} skip={skip_cnt}")

def process_all_lines(line_urls):
    dic = {}

    for url in line_urls:
        process_single_line(url, dic)

    return dic


In [ ]:
ensen_html = download("https://www.example-realestate-benchmark.jp/chintai/soba/tokyo/ensen/")

line_urls = extract_line_urls_from_ensen(ensen_html)
print("Total line URLs:", len(line_urls))

station_rent_dict = process_all_lines(line_urls)

df_station = pd.DataFrame(
    [(station, yen) for station, yen in station_rent_dict.items()],
    columns=["station", "avg_rent_yen"]
)

df_station.to_csv("tokyo_station_avg_rent.csv", index=False, encoding="utf-8-sig")

print("Saved to tokyo_station_avg_rent.csv")

Line URLs extracted: 82  dup=0
Total line URLs: 82
en_yamanotesen new=30 dup=0 skip=0
en_keihintohokusen new=23 dup=15 skip=0
en_tokaidohonsen new=14 dup=6 skip=0
en_jobansen new=45 dup=5 skip=0
en_nambusen new=29 dup=1 skip=0
en_yokohamasen new=19 dup=1 skip=0
en_yokosukasen new=12 dup=7 skip=0
en_chuosen new=28 dup=6 skip=0
en_omesen new=23 dup=1 skip=0
en_itsukaichisen new=6 dup=1 skip=0
en_musashinosen new=23 dup=3 skip=0
en_hachikosen new=22 dup=2 skip=0
en_saikyosen new=13 dup=9 skip=0
en_takasakisen new=19 dup=6 skip=0
en_utsunomiyasen new=20 dup=7 skip=0
en_sobusen new=19 dup=20 skip=0
en_sobukaisokusen new=2 dup=8 skip=0
en_keiyosen new=17 dup=2 skip=0
en_shonanshinjukulineutsunomiyayokosukasen new=0 dup=33 skip=0
en_shonanshinjukulinetakasakitokaidosen new=4 dup=39 skip=0
en_tokaidoshinkansen new=0 dup=4 skip=0
en_joetsushinkansen new=2 dup=5 skip=0
en_hokurikushinkansen new=1 dup=6 skip=0
en_tohokushinkansen new=1 dup=5 skip=0
en_ginzasen new=15 dup=4 skip=0
en_marunouchisen

In [ ]:
df_station = pd.read_csv("tokyo_station_avg_rent.csv", encoding="utf-8-sig")

print(df_station.head())
print(df_station.shape)

station_rent_map = {station: int(rent) for station, rent in zip(df_station.station, df_station.avg_rent_yen)}

print(station_rent_map)

  station  avg_rent_yen
0      東京         98000
1     有楽町        123000
2      新橋        119000
3     浜松町        112000
4      田町        103000
(1101, 2)
{'東京': 98000, '有楽町': 123000, '新橋': 119000, '浜松町': 112000, '田町': 103000, '高輪ゲートウェイ': 95000, '品川': 94000, '大崎': 85000, '五反田': 90000, '目黒': 100000, '恵比寿': 114000, '渋谷': 114000, '原宿': 119000, '代々木': 100000, '新宿': 84000, '新大久保': 80000, '高田馬場': 77000, '目白': 74000, '池袋': 75000, '大塚': 76000, '巣鴨': 78000, '駒込': 75000, '田端': 73000, '西日暮里': 75000, '日暮里': 79000, '鶯谷': 79000, '上野': 85000, '御徒町': 89000, '秋葉原': 96000, '神田': 106000, '赤羽': 70000, '東十条': 66000, '王子': 69000, '上中里': 70000, '大井町': 81000, '大森': 73000, '蒲田': 75000, '川崎': 64000, '鶴見': 59000, '新子安': 50000, '東神奈川': 63000, '横浜': 68000, '桜木町': 70000, '関内': 69000, '石川町': 62000, '山手': 52000, '根岸': 43000, '磯子': 40000, '新杉田': 45000, '洋光台': 40000, '港南台': 46000, '本郷台': 50000, '大船': 55000, '戸塚': 52000, '藤沢': 60000, '辻堂': 58000, '茅ケ崎': 54000, '平塚': 42000, '大磯': 32000, '二宮': 26000, '国府津': 46000, '鴨宮': 45

In [ ]:
def parse_total_count(html):
    """
    Robustly extract total count.
    Prefer the <span class="totalCount">395</span> form, fallback to general patterns.
    """
    if not html:
        raise ValueError("Empty HTML passed to parse_total_count.")

    # 1) Preferred: <span class="totalCount">395</span>
    m = re.search(r'class=["\']totalCount["\'][^>]*>(\d+)</span>', html)
    if m:
        return int(m.group(1))

    # # 2) Fallback: e.g. "<span class=\"totalCount\">395</span>件" (span and then 件)
    # m2 = re.search(r'<span[^>]*>(\d+)</span>\s*件', html)
    # if m2:
    #     return int(m2.group(1))

    # # 3) Fallback: any "◯件" like "395件の..."
    # m3 = re.search(r'(\d+)\s*件', html)
    # if m3:
    #     return int(m3.group(1))

    raise ValueError("Cannot find total count (◯件) in HTML.")


def extract_ids(html):
    """
    Extract IDs from DISP_BUKKEN_KEY_LIST.push("000000...") lines.
    Returns list of id strings.
    """
    if not html:
        return []
    ids = re.findall(r'DISP_BUKKEN_KEY_LIST\.push\(["\'](\d+)["\']\)', html)
    return ids


def get_all_ids(area_code):
    """
    Given area_code like "101010044",
    return a list of ALL bk IDs appearing in that area's list pages.
    Using p=10 (100 items/page).
    """
    base = f"https://www.example-realestate-portal-a.jp/tokyo/ensen/{area_code}/list/  # Anonymized URL (Real Estate Portal A)"
    p_param = "?p=10&o="

    # 1. First page
    url_first = base + p_param
    html_first = download(url_first)
    if not html_first:
        raise RuntimeError("Failed to download first page.")

    # 2. Total count and pages
    total = parse_total_count(html_first)
    pages = math.ceil(total / 100)

    # 3. Extract ids from first page
    ids = extract_ids(html_first)

    # 4. Loop pages 2..pages
    for i in range(2, pages + 1):
        url = base + f"?p=10&o=&i={i}"
        html = download(url)
        if not html:
            print(f"[WARN] Failed to download page {i}. Skipping.")
            continue
        page_ids = extract_ids(html)
        if not page_ids:
            # fallback: if page gives "物件が見つかりませんでした" stop early
            if "物件が見つかりませんでした" in html:
                print(f"[INFO] page {i} reports no results. Stopping.")
                break
            else:
                print(f"[WARN] No DISP_BUKKEN_KEY_LIST on page {i}.")
        ids += page_ids

    # 5. Dedupe and validate
    ids = list(dict.fromkeys(ids))  # preserve order, remove duplicates

    if len(ids) != total:
        print(f"[WARN] Expected {total} ids, but got {len(ids)}. (Still returning what we have.)")

    return ids


  # ============ Raw Data Class ============

@dataclass
class PortalARawData:
    id: str
    building_type: Optional[str]
    structure: Optional[str]
    built_year: Optional[str]
    layout: Optional[str]
    floor: Optional[str]
    orientation: Optional[str]
    area: Optional[str]
    rent: Optional[str]
    maintenance: Optional[str]
    deposit: Optional[str]
    reikin: Optional[str]
    nearest_station_minutes: Optional[int]
    nearest_station_name: Optional[str]
    features: Dict[str, Optional[bool]]


def find_first_match(pattern):
    """Return the first capture group of `pattern` from global html."""
    m = re.search(pattern, html, flags=re.DOTALL)
    return m.group(1).strip() if m else None


def find_label(label):
    """Extract <td>...</td> following <th>LABEL</th> using global html."""
    pattern = rf'<th>{re.escape(label)}</th>\s*<td>([^<]+)</td>'
    return find_first_match(pattern)

def detect_feature(keyword_jp):
    """
    Return True/False/None based on:
      <li>                → True
      <li class="not">    → False
      not found           → None
    keyword_jp = String within "span"（eg. "ペット相談可"）
    """
    pattern = rf'<li[^>]*>(?:(?!</li>).)*?<span>{re.escape(keyword_jp)}</span>(?:(?!</li>).)*?</li>'
    m = re.search(pattern, html, flags=re.DOTALL)

    if not m:
        return None

    li_block = m.group(0)

    if 'class="not"' in li_block:
        return False
    else:
        return True

# ============ Main Function ============

def get_raw_data(bk_id: str) -> PortalARawData:
    global html

    # -----------------------------------------
    # (1) Download HTML
    # -----------------------------------------
    url = f"https://www.example-realestate-portal-a.jp/detail/Detail.do?bk={bk_id}  # Anonymized URL (Portal A)"
    html = requests.get(url).text

    # -----------------------------------------
    # (2) Extract building_type / structure before truncation
    # -----------------------------------------
    building_type = None
    structure = None

    m_struct = re.search(
        r'<th>建物種別/構造</th>\s*<td>([^<]+)</td>',
        html
    )

    if m_struct:
        combined = m_struct.group(1).strip()

        if "/" in combined:
            building_type, structure = map(str.strip, combined.split("/", 1))

    # -----------------------------------------
    # (3) Truncation
    # -----------------------------------------
    cut_marker = "<th>共有スペースの特徴・設備</th>"
    pos = html.find(cut_marker)
    if pos != -1:
        html = html[:pos]

    # -----------------------------------------
    # (4) Extract rent & maintenance
    # -----------------------------------------
    rent = find_first_match(
        r'<span class="price">(\d+(?:\.\d+)?)<span>万円</span></span>'
    )

    maintenance = find_first_match(
        r'<br>\s*([\d,]+)円'
    )

    # -----------------------------------------
    # (5) Find_label in Summary table fields
    # -----------------------------------------
    built_year  = find_label("築年")
    layout      = find_label("間取り")
    floor       = find_label("階/階建")
    orientation = find_label("向き")
    area        = find_label("広さ")
    deposit     = find_label("敷金/保証金")
    reikin      = find_label("礼金/償却")

    # -----------------------------------------
    # (6) Extract nearest station from <th>交通</th> block
    # -----------------------------------------

    BUS_TO_WALK = 4

    # 1. Extract the traffic <td>...</td> block
    m_traffic = re.search(
        r'<th[^>]*>交通</th>\s*<td[^>]*>(.*?)</td>',
        html,
        flags=re.DOTALL
    )

    nearest_station_minutes = None
    nearest_station_name = None

    if m_traffic:
        traffic_html = m_traffic.group(1)

        # 2. Extract each <li>...</li> (each station entry)
        li_blocks = re.findall(r'<li>(.*?)</li>', traffic_html, flags=re.DOTALL)

        candidates = []

        for li in li_blocks:
            # 3. Extract the second <a>…</a> tag → station name
            a_tags = re.findall(r'<a[^>]*>(.*?)</a>', li)

            if len(a_tags) < 2:
                continue  # abnormal format, skip

            station_raw = a_tags[1].strip()   # e.g. "東小金井駅"
            station_clean = station_raw.rstrip("駅")  # remove last "駅"

            # 4. Parse time: 徒歩 or バス + 停歩
            walk_m = re.search(r'徒歩(\d+)分', li)
            bus_m  = re.search(r'バス(\d+)分', li)
            step_m = re.search(r'停歩(\d+)分', li)

            eq = None
            if walk_m:
                eq = int(walk_m.group(1))
            elif bus_m and step_m:
                eq = int(bus_m.group(1)) * BUS_TO_WALK + int(step_m.group(1))

            # If time parsed correctly → store candidate
            if eq is not None:
                candidates.append((eq, station_clean))

        # 5. Pick nearest station
        if candidates:
            candidates.sort(key=lambda x: x[0])
            nearest_station_minutes, nearest_station_name = candidates[0]

    # -----------------------------------------
    # (7) Features extraction
    # -----------------------------------------
    features = {}

    features["floor_2_or_above"]      = detect_feature("2階以上")
    features["bath_toilet_separated"] = detect_feature("バス・トイレ別")
    features["independent_washstand"] = detect_feature("独立洗面台")
    features["air_conditioner"]       = detect_feature("エアコン")
    features["indoor_laundry_space"]  = detect_feature("室内洗濯機置場")
    features["auto_lock"]             = detect_feature("オートロック")
    features["pet_allowed"]           = detect_feature("ペット相談可")
    features["parking"]               = detect_feature("駐車場")
    features["delivery_box"]          = detect_feature("宅配ボックス")
    features["warm_water_toilet"]     = detect_feature("温水洗浄便座")
    features["security_camera"]       = detect_feature("防犯カメラ")
    features["bath_dryer"]            = detect_feature("浴室乾燥機")

    # -----------------------------------------
    # (8) corner_room and internet_free
    # -----------------------------------------
    m_corner = re.search(
        r'<th>お部屋の特徴</th>\s*<td>(.*?)</td>',
        html,
        flags=re.DOTALL
    )
    corner_room = ("角部屋" in m_corner.group(1)) if m_corner else False
    features["corner_room"] = corner_room
    features["internet_free"] = bool(re.search(r'インターネット利用無料', html))

    # -----------------------------------------
    # (9) Return Object
    # -----------------------------------------
    return PortalARawData(
        id=bk_id,
        building_type=building_type,
        structure=structure,
        built_year=built_year,
        layout=layout,
        floor=floor,
        orientation=orientation,
        area=area,
        rent=rent,
        maintenance=maintenance,
        deposit=deposit,
        reikin=reikin,
        nearest_station_minutes=nearest_station_minutes,
        nearest_station_name=nearest_station_name,
        features=features
    )
# --------------------------------------------------------
# Helper functions
# --------------------------------------------------------

def parse_floor_field(text: Optional[str]):
    """
    Convert:
        "2階/3階建"       → floor_min=2, floor_max=2, floor_total=3
        "1-2階/5階建"     → floor_min=1, floor_max=2, floor_total=5
    """
    if not text:
        return None, None, None

    m = re.match(r'(.+?)階/(\d+)階建', text)
    if not m:
        return None, None, None

    raw_floor = m.group(1)
    total = int(m.group(2))

    # Case 1: simple "3階"
    if re.match(r'^\d+$', raw_floor):
        f = int(raw_floor)
        return f, f, total

    # Case 2: "1-2階"
    m2 = re.match(r'^(\d+)-(\d+)$', raw_floor)
    if m2:
        f1 = int(m2.group(1))
        f2 = int(m2.group(2))
        return f1, f2, total

    return None, None, total


def parse_money(text: Optional[str]):
    """
    Parse a money string from Portal A  into two integer amounts (left, right),
    each in Japanese yen.

    Examples:
        "6万/--"         → (60000, 0)
        "--/10万円"      → (0, 100000)
        "4.1万/4.1万"    → (41000, 41000)
        "50000円/--"     → (50000, 0)
        "--"             → (0, 0)
        "10万円"         → (100000, 0)
        "12万円/8万円"   → (120000, 80000)
        "--/50,000円"    → (0, 50000)
    """

    def parse_one(x: Optional[str]) -> int:
        """
        Parse a single amount fragment and return the value in yen.

        Accepts formats such as:
            "6万", "4.1万", "10万円", "50000円", "50,000円", "--".
        Never raises exceptions. Returns 0 for invalid or missing values.
        """
        if not x:
            return 0

        x = x.strip()

        # Missing / Not applicable
        if x == "--":
            return 0

        # Normalize "万円" into "万"
        # E.g., "10万円" -> "10万"
        x = x.replace("万円", "万")

        # Case 1: values expressed in "万" (ten-thousand-yen units)
        if "万" in x:
            num_str = x.replace("万", "").strip()
            # Clean unwanted characters like 円 or commas
            num_str = num_str.replace("円", "").replace(",", "")
            try:
                num = float(num_str)
                return int(num * 10000)
            except:
                return 0

        # Case 2: plain yen amount like "50000円" or "50,000円"
        plain = x.replace("円", "").replace(",", "")
        if plain.isdigit():
            return int(plain)

        # Fallback if none of the patterns match
        return 0

    # ==========================
    # Main parsing logic
    # ==========================

    if not text or text.strip() == "--":
        return 0, 0

    # If no slash "/", interpret the value as a single amount
    if "/" not in text:
        return parse_one(text), 0

    left, right = text.split("/", 1)
    return parse_one(left), parse_one(right)

def yen_from_rent_man(text: Optional[str]) -> int:
    """
    rent field: "8.2" → 82000
    """
    if not text:
        return None

    text = text.strip()
    try:
        return int(float(text) * 10000)
    except:
        return None


def parse_area(text: Optional[str]) -> Optional[float]:
    """
    "30.48㎡" → 30.48
    """
    if not text:
        return None

    m = re.match(r'([\d\.]+)', text)
    if m:
        return float(m.group(1))
    return None


def parse_age(text: Optional[str]) -> Optional[int]:
    """
    Handle:
        築18年  → 18
        築40年  → 40
        築1年未満 → 0
        新築 → 0
        築0年 → 0
    """
    if not text:
        return None

    text = text.strip()

    # 新築
    if text == "新築":
        return 0

    # 1年未満
    if "年未満" in text:
        return 0

    # 築○年
    m = re.match(r'築(\d+)年', text)
    if m:
        return int(m.group(1))

    return None



def bool_to_int(x: Optional[bool]) -> Optional[int]:
    if x is True:
        return 1
    if x is False:
        return 0
    return None


# ========================================================
#                CleanData Main Function
# ========================================================

def clean_data(raw: PortalARawData) -> Dict:
    """
    Convert PortalARawData → CleanedDataStage2 (ML-ready dict)
    """

    # ---------------------------------------------
    # Stage 1 : Raw → Structured
    # ---------------------------------------------
    rent_yen = yen_from_rent_man(raw.rent)
    maintenance_yen = 0
    if raw.maintenance:
        maintenance_yen = int(raw.maintenance.replace(",", "").replace("円", "") or 0)

    area_m2 = parse_area(raw.area)
    building_age = parse_age(raw.built_year)

    floor_min, floor_max, floor_total = parse_floor_field(raw.floor)

    # Four amounts

    deposit_yen, guarantee_yen = parse_money(raw.deposit)
    reikin_yen, shoukyaku_yen  = parse_money(raw.reikin)


    # ---------------------------------------------
    # Stage 2 : Structured → ML Features
    # ---------------------------------------------

    # Ratio fields
    denom = rent_yen or 1   # Preventing division errors where rent=0
    deposit_ratio = deposit_yen / denom
    guarantee_ratio = guarantee_yen / denom
    reikin_ratio = reikin_yen / denom
    shoukyaku_ratio = shoukyaku_yen / denom

    # boolean features → 0/1/None
    feature_numeric = {
        k: bool_to_int(v) for k, v in raw.features.items()
    }

    station_name = raw.nearest_station_name
    nearest_station_avg_rent = None

    if station_name:
        try:
            # First try original
            nearest_station_avg_rent = station_rent_map[station_name]
        except KeyError:
            # Fallback replace ケ → ヶ
            alt = station_name.replace("ケ", "ヶ")
            if alt in station_rent_map:
                nearest_station_avg_rent = station_rent_map[alt]
            else:
                nearest_station_avg_rent = None
    # ---------------------------------------------
    # Final dictionary (DataFrame-friendly)
    # ---------------------------------------------

    data = {
        # categorical (Keep the original string)
        "id": raw.id,
        "building_type": raw.building_type,
        "structure": raw.structure,
        "layout": raw.layout,
        "orientation": raw.orientation,

        # numeric

        "nearest_station_avg_rent": nearest_station_avg_rent,
        "monthly_cost_yen": rent_yen + maintenance_yen,
        "area_m2": area_m2,
        "building_age": building_age,

        "floor_min": floor_min,
        "floor_max": floor_max,
        "floor_total": floor_total,

        "nearest_station_minutes": raw.nearest_station_minutes,

        # ratio
        "deposit_ratio": deposit_ratio,
        "guarantee_ratio": guarantee_ratio,
        "reikin_ratio": reikin_ratio,
        "shoukyaku_ratio": shoukyaku_ratio,
    }

    # Add all Boolean numeric features
    data.update(feature_numeric)

    return data

In [ ]:
def find_problem_ids(df):
    """
    Return a list of bk IDs whose cleaned data contains ANY NaN.
    """
    nan_rows = df[df.isna().any(axis=1)]

    problem_ids = nan_rows["id"].unique().tolist()

    print(f"Found {len(problem_ids)} problematic rows.")
    return problem_ids

In [ ]:
def build_area_dataset(area_code, station_name):
    """
    Generate a cleaned dataset CSV for a given area_code and station_name.

    Output CSV →  "{area_code}_{station_name}.csv"

    Steps:
    1. get all bk ids
    2. fetch raw HTML for each property
    3. convert to PortalARawData
    4. clean into ML-friendly dict
    5. build DataFrame
    6. detect problematic ids (print only)
    7. save CSV
    """

    print(f"\n=== Building dataset for area_code: {area_code} ({station_name}) ===")

    # Step 1 — Get all room IDs
    ids = get_all_ids(area_code)
    print(f"Total IDs found: {len(ids)}")
    print("First 10 IDs:", ids[:10])

    # Step 2 — Fetch & clean
    data_list = []
    for bk in ids:
        try:
            raw = get_raw_data(bk)
            clean = clean_data(raw)
            data_list.append(clean)
        except Exception as e:
            print(f"[ERROR] Failed to process bk={bk}: {e}")

    # Step 3 — Build DataFrame
    df = pd.DataFrame(data_list)
    print(f"DataFrame shape: {df.shape}")

    # Step 4 — Detect problematic IDs (print only)
    try:
        problem_ids = find_problem_ids(df)
        print(f"Problem IDs found: {len(problem_ids)}")
        print(problem_ids)
    except Exception as e:
        print(f"find_problem_ids() not executed: {e}")

    # Step 5 — Save CSV with new naming rule
    csv_name = f"{area_code}_{station_name}.csv"
    df.to_csv(csv_name, index=False, encoding="utf-8")
    print(f"Saved dataset to: {csv_name}")

    return df

In [ ]:
df = build_area_dataset("101010044", "higashi-koganei")


=== Building dataset for area_code: 101010044 (higashi-koganei) ===
Total IDs found: 381
First 10 IDs: ['000000049282000006', '000000020726963008', '000000049169309001', '000000049398948005', '000000081868839001', '000000049313425020', '000000049313425013', '000000081695829005', '000000049716691002', '000000049911495006']
DataFrame shape: (381, 31)
Found 0 problematic rows.
Problem IDs found: 0
[]
Saved dataset to: 101010044_higashi-koganei.csv


In [ ]:
# def print_raw_by_ids(id_list):
#     for bk in id_list:
#         try:
#             raw = get_raw_data(bk)
#             print("\n==============================")
#             print(f"RAW DATA for BK = {bk}")
#             print("==============================")
#             print(raw)
#         except Exception as e:
#             print(f"[ERROR] Cannot fetch raw data for {bk}: {e}")
# problem_ids = ['000000081103232004', '000000020164747001', '000000049101448000']
# print_raw_by_ids(problem_ids)


In [ ]:
df = build_area_dataset("102065004", "shin-koganei")


=== Building dataset for area_code: 102065004 (shin-koganei) ===
Total IDs found: 332
First 10 IDs: ['000000049161593002', '000000049858604002', '000000081697598002', '000000049911495006', '000000081695829005', '000000081759024002', '000000049716691002', '000000049313425020', '000000049313425013', '000000049262973001']
DataFrame shape: (332, 31)
Found 0 problematic rows.
Problem IDs found: 0
[]
Saved dataset to: 102065004_shin-koganei.csv


In [ ]:
df = build_area_dataset("102066036", "hana-koganei")


=== Building dataset for area_code: 102066036 (hana-koganei) ===
Total IDs found: 246
First 10 IDs: ['000000017315126004', '000000017845824002', '000000017716996005', '000000017876383007', '000000017646696001', '000000017497011002', '000000017010876007', '000000017504455002', '000000017712746003', '000000017199179002']
DataFrame shape: (246, 31)
Found 0 problematic rows.
Problem IDs found: 0
[]
Saved dataset to: 102066036_hana-koganei.csv


In [ ]:
df = build_area_dataset("101010046", "musashi-koganei")


=== Building dataset for area_code: 101010046 (musashi-koganei) ===
Total IDs found: 388
First 10 IDs: ['000000049267151001', '000000049269040002', '000000049327834004', '000000049327834007', '000000049723007002', '000000049729504004', '000000049049588001', '000000049683198007', '000000027094921001', '000000049330385003']
DataFrame shape: (388, 31)
Found 0 problematic rows.
Problem IDs found: 0
[]
Saved dataset to: 101010046_musashi-koganei.csv


In [ ]:
df = build_area_dataset("101010042", "musashisakai")


=== Building dataset for area_code: 101010042 (musashisakai) ===
Total IDs found: 408
First 10 IDs: ['000000081794138008', '000000081794138043', '000000081794138041', '000000081794138040', '000000081794138025', '000000081172883001', '000000081402638001', '000000081305325004', '000000081166177004', '000000081166177012']
DataFrame shape: (408, 31)
Found 0 problematic rows.
Problem IDs found: 0
[]
Saved dataset to: 101010042_musashisakai.csv


In [ ]:
df = build_area_dataset("101010060", "hachioji")


=== Building dataset for area_code: 101010060 (hachioji) ===
[WARN] Expected 384 ids, but got 306. (Still returning what we have.)
Total IDs found: 306
First 10 IDs: ['000000029878617005', '000000029878617007', '000000029154174001', '000000029300629003', '000000029878337004', '000000029451191005', '000000029158109006', '000000029158109015', '000000029677325006', '000000029168219001']
DataFrame shape: (306, 31)
Found 0 problematic rows.
Problem IDs found: 0
[]
Saved dataset to: 101010060_hachioji.csv


In [ ]:
def load_all_station_datasets():
    """
    Scan current directory for CSV files matching:
        <area_code>_<station_name>.csv
    Example filenames:
        101010044_higashi-koganei.csv
        102065004_shin-koganei.csv

    Returns:
        dict: {
            "higashi-koganei": DataFrame,
            "shin-koganei": DataFrame,
            ...
        }
    """
    dfs = {}
    pattern = re.compile(r'^(\d+)_([A-Za-z0-9\-]+)\.csv$')

    for file in glob.glob("*.csv"):
        m = pattern.match(file)
        if not m:
            continue

        station_name = m.group(2)  # extract station name

        try:
            df = pd.read_csv(file, encoding="utf-8")
            dfs[station_name] = df
            print(f"Loaded: {file} → key='{station_name}', shape={df.shape}")
        except Exception as e:
            print(f"[ERROR] Cannot load {file}: {e}")

    print(f"\nTotal datasets loaded: {len(dfs)}")
    return dfs

In [ ]:
df = load_all_station_datasets()

Loaded: 101010044_higashi-koganei.csv → key='higashi-koganei', shape=(381, 31)
Loaded: 101010060_hachioji.csv → key='hachioji', shape=(306, 31)
Loaded: 101010046_musashi-koganei.csv → key='musashi-koganei', shape=(388, 31)
Loaded: 102065004_shin-koganei.csv → key='shin-koganei', shape=(332, 31)
Loaded: 101010042_musashisakai.csv → key='musashisakai', shape=(408, 31)
Loaded: 102066036_hana-koganei.csv → key='hana-koganei', shape=(246, 31)

Total datasets loaded: 6


In [ ]:
df.keys()

dict_keys(['higashi-koganei', 'hachioji', 'musashi-koganei', 'shin-koganei', 'musashisakai', 'hana-koganei'])

In [ ]:
df["shin-koganei"].head()

,id,building_type,structure,layout,orientation,nearest_station_avg_rent,monthly_cost_yen,area_m2,building_age,floor_min,...,indoor_laundry_space,auto_lock,pet_allowed,parking,delivery_box,warm_water_toilet,security_camera,bath_dryer,corner_room,internet_free
0,49161593002,コーポ,木造,1K,南,55000,65000,24.80,27,1,...,1,0,0,0,0,0,0,0,0,1
1,49858604002,コーポ,木造,1K,南,55000,41000,17.01,36,2,...,1,0,0,0,0,0,0,0,0,0
2,81697598002,マンション,鉄骨造,1K,南,55000,67000,24.00,22,1,...,1,0,0,0,0,1,0,0,0,0
3,49911495006,コーポ,木造,1K,西,55000,53000,20.28,33,1,...,1,0,0,1,0,0,0,0,0,0
4,81695829005,ハイツ,軽量鉄骨造,1K,南,55000,83000,26.72,21,1,...,1,1,0,0,0,0,0,1,0,0


In [ ]:
import pandas as pd

def merge_all_stations(df_dict):

    df_list = []
    for station_name, df in df_dict.items():
        df = df.copy()
        df["source_station"] = station_name
        df_list.append(df)

    df_all = pd.concat(df_list, ignore_index=True)
    print(f"Merged shape: {df_all.shape}")

    categorical_cols = [
        "building_type",
        "structure",
        "layout",
        "orientation",
    ]

    df_all = pd.get_dummies(df_all, columns=categorical_cols, dummy_na=True)

    # ---- Step 4: fill NA ----
    df_all = df_all.fillna(0)

    return df_all


In [ ]:
df_all_ready = merge_all_stations(df)
print(df_all_ready.shape)
df_all_ready.head()

Merged shape: (2061, 32)
(2061, 73)


,id,nearest_station_avg_rent,monthly_cost_yen,area_m2,building_age,floor_min,floor_max,floor_total,nearest_station_minutes,deposit_ratio,...,layout_nan,orientation_北,orientation_北東,orientation_北西,orientation_南,orientation_南東,orientation_南西,orientation_東,orientation_西,orientation_nan
0,49282000006,58000,66000,19.30,28,4,4,6,2,1.0,...,False,False,False,False,False,False,False,False,True,False
1,20726963008,58000,50000,20.00,34,1,1,2,3,1.0,...,False,False,False,False,True,False,False,False,False,False
2,49169309001,58000,130000,42.70,30,2,2,2,3,1.0,...,False,False,False,False,False,False,False,True,False,False
3,49398948005,58000,62000,22.36,35,4,4,4,3,1.0,...,False,False,False,False,False,False,False,True,False,False
4,81868839001,58000,61000,20.70,28,1,1,2,7,0.0,...,False,False,False,False,False,False,False,False,True,False


In [112]:
def train_and_evaluate(
        df_all,
        train_stations,
        use_log_target=False
    ):

    # Convert single string to list
    if isinstance(train_stations, str):
        train_stations = [train_stations]

    # ---------------------------
    # 1. Select training set
    # ---------------------------
    train_df = df_all[df_all["source_station"].isin(train_stations)].copy()

    # Prepare y
    if use_log_target:
        train_df["target"] = np.log1p(train_df["monthly_cost_yen"])
    else:
        train_df["target"] = train_df["monthly_cost_yen"]

    # Prepare X
    X_train = train_df.drop(columns=["id", "monthly_cost_yen", "source_station", "target"])
    y_train = train_df["target"]

    # ---------------------------
    # 2. XGBRegressor
    # ---------------------------
    model = XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    # ---------------------------
    # 3. Train
    # ---------------------------
    model.fit(X_train, y_train)

    # ---------------------------
    # 4. Evaluate per station
    # ---------------------------
    results = []

    for station in df_all["source_station"].unique():

        test_df = df_all[df_all["source_station"] == station].copy()

        # remove leakage
        test_df = test_df[~test_df["id"].isin(train_df["id"])]

        if len(test_df) == 0:
            continue

        # target
        if use_log_target:
            test_df["target"] = np.log1p(test_df["monthly_cost_yen"])
        else:
            test_df["target"] = test_df["monthly_cost_yen"]

        y_test = test_df["target"]

        # align columns
        X_test = test_df.drop(columns=["id", "monthly_cost_yen", "source_station", "target"])
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        y_pred = model.predict(X_test)

        if use_log_target:
            rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
        else:
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        r2 = r2_score(y_test, y_pred)
        mape = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred)) / np.expm1(y_test))) \
            if use_log_target else \
            np.mean(np.abs((y_test - y_pred) / y_test))

        results.append([station, len(test_df), rmse, r2, mape])

    # ---------------------------
    # 5. Evaluate global test set
    # ---------------------------
    remaining_df = df_all[~df_all["id"].isin(train_df["id"])].copy()

    if use_log_target:
        remaining_df["target"] = np.log1p(remaining_df["monthly_cost_yen"])
    else:
        remaining_df["target"] = remaining_df["monthly_cost_yen"]

    y_test = remaining_df["target"]
    X_test = remaining_df.drop(columns=["id", "monthly_cost_yen", "source_station", "target"])
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    y_pred = model.predict(X_test)

    if use_log_target:
        rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
    else:
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((np.expm1(y_test) - np.expm1(y_pred)) / np.expm1(y_test))) \
        if use_log_target else \
        np.mean(np.abs((y_test - y_pred) / y_test))

    results.append(["ALL", len(remaining_df), rmse, r2, mape])

    # ---------------------------
    # 6. Build summary table
    # ---------------------------
    summary = pd.DataFrame(results, columns=["station", "samples", "RMSE", "R2", "MAPE"])
    return summary, model


In [118]:
summary, model = train_and_evaluate(df_all_ready, "higashi-koganei")
summary

,station,samples,RMSE,R2,MAPE
0,hachioji,306,18057.159024,0.343168,0.284168
1,musashi-koganei,106,10859.669977,0.878357,0.136545
2,shin-koganei,35,8795.713956,0.862686,0.094611
3,musashisakai,85,18543.451243,0.804546,0.125707
4,hana-koganei,195,14185.914704,0.750403,0.141721
5,ALL,727,15885.298864,0.728186,0.196783


In [119]:
summary, model = train_and_evaluate(df_all_ready, "higashi-koganei", True)
summary

,station,samples,RMSE,R2,MAPE
0,hachioji,306,15437.548175,0.457320,0.266592
1,musashi-koganei,106,12091.322985,0.832918,0.139036
2,shin-koganei,35,9082.889701,0.847482,0.097725
3,musashisakai,85,18542.691625,0.791542,0.120473
4,hana-koganei,195,11300.175185,0.817207,0.131719
5,ALL,727,14143.883260,0.730049,0.186604


In [117]:
summary, model = train_and_evaluate(df_all_ready, ["higashi-koganei", "musashi-koganei"])
summary

,station,samples,RMSE,R2,MAPE
0,hachioji,306,15968.036573,0.486360,0.247185
1,shin-koganei,35,10033.683671,0.821313,0.103972
2,musashisakai,85,18421.126133,0.807117,0.121560
3,hana-koganei,189,12491.423618,0.811124,0.128124
4,ALL,615,15081.427519,0.754091,0.185082


In [120]:
summary, model = train_and_evaluate(df_all_ready, ["higashi-koganei", "musashi-koganei"], True)
summary

,station,samples,RMSE,R2,MAPE
0,hachioji,306,14279.898520,0.541541,0.235966
1,shin-koganei,35,9269.706924,0.857243,0.099806
2,musashisakai,85,19548.865142,0.782000,0.125644
3,hana-koganei,189,11286.745630,0.829297,0.129011
4,ALL,615,14082.559950,0.746593,0.180100


In [121]:
train_stations = [
    "higashi-koganei",
    "shin-koganei",
    "hana-koganei",
    "musashi-koganei",
    "musashisakai",
]

summary, model = train_and_evaluate(df_all_ready, train_stations, False)
summary

,station,samples,RMSE,R2,MAPE
0,hachioji,306,12079.333094,0.706072,0.199844
1,ALL,306,12079.333094,0.706072,0.199844


In [122]:
summary, model = train_and_evaluate(df_all_ready, train_stations, True)
summary

,station,samples,RMSE,R2,MAPE
0,hachioji,306,11721.588845,0.652935,0.187083
1,ALL,306,11721.588845,0.652935,0.187083
